# 2. Simulaciones: identificabilidad, robustez, análisis de poder
No necesita datos reales. Escribe los scripts y los corre. Las celdas de `--mice 60` tardan; baja el número para una prueba rápida.

In [ ]:
!pip -q install numba scipy pandas matplotlib numpy

In [ ]:
%%writefile models.py
"""Trial-level models of lick / withhold behaviour in a change-detection task.

All models share one observation model and the same nuisance terms; they differ ONLY in how
the value difference and the inverse temperature evolve:

    logit P(lick_t) = beta_t * (V_lick - V_withhold) + b + w * sat_t
    sat_t           = cumulative reward before trial t, divided by 100  (satiety / motivation proxy)

bias      no learning: logit = b + w*sat
q         TD learning, fixed alpha, fixed beta                       (= DMA with the gate switched off)
ph        Pearce-Hall style: surprise gates the LEARNING RATE
dma       surprise gates the INVERSE TEMPERATURE through a bAP/HCN conductance g_t
          (the discrete DMA mapping of the reference implementation); alpha, beta_base, lambda, tau_hcn free
dma_const same equations with the constants published for DMA, only (b, w) fitted

Teacher forcing: the observed action drives learning; we score its probability.
Reward = 1 only for a lick on a go trial (as in the task); delta uses the chosen action's value
exactly like the reference implementation.
"""
from __future__ import annotations

import math

import numpy as np

try:  # numba is optional; without it everything still runs, only much slower
    from numba import njit
except Exception:  # pragma: no cover
    def njit(*a, **k):
        if len(a) == 1 and callable(a[0]):
            return a[0]
        return lambda f: f

# Published DMA constants (reference implementation)
ALPHA_C, TAU_C, GMAX, VMAX, LAMBDA_C, BETA_C = 0.15, 45.0, 1.0, 1.0, 0.8, 5.5

MODELS = ["bias", "q", "ph", "dma", "dma_const", "ck", "epi", "hmm", "hmmck", "qck", "dmack"]
MID = {m: i for i, m in enumerate(MODELS)}

# name, bounds, starting points
SPECS = {
    "bias": (["b", "w", "s"], [(-14, 14), (-8, 8), (0, 12)], [[0.0, 0.0, 3.0]]),
    "q": (["alpha", "beta", "b", "w", "s"], [(0.01, 0.9), (0.2, 15), (-14, 14), (-8, 8), (0, 12)],
          [[0.15, 4.0, -4.0, 0.0, 3.0], [0.4, 8.0, -7.0, -1.0, 4.0]]),
    "ph": (["alpha0", "kappa", "eta", "beta", "b", "w", "s"],
           [(0.01, 0.9), (0.0, 5.0), (0.01, 0.9), (0.2, 15), (-14, 14), (-8, 8), (0, 12)],
           [[0.15, 1.0, 0.2, 4.0, -4.0, 0.0, 3.0], [0.3, 0.3, 0.5, 8.0, -7.0, -1.0, 4.0]]),
    "dma": (["alpha", "beta_base", "lam", "tau_hcn", "b", "w", "s"],
            [(0.01, 0.9), (0.2, 15), (0.0, 1.5), (3.0, 300.0), (-14, 14), (-8, 8), (0, 12)],
            [[0.15, 5.5, 0.8, 45.0, -5.0, 0.0, 3.0], [0.3, 8.0, 0.4, 20.0, -7.0, -1.0, 4.0]]),
    "dma_const": (["b", "w", "s"], [(-14, 14), (-8, 8), (0, 12)], [[-5.0, 0.0, 3.0], [-7.0, -1.0, 4.0]]),
    # episodic-retrieval models (adapted from the reference implementation's episodic memory):
    # ck  : recency-weighted retrieval of past choices made under the same cue (a plain choice kernel; control)
    # epi : ck plus retrieval of "problem" engrams (a lick that was not rewarded) under the same cue
    "ck": (["rho", "g1", "b", "w", "s"], [(0.3, 0.999), (-12, 12), (-14, 14), (-8, 8), (0, 12)],
           [[0.9, 1.0, -4.0, 0.0, 3.0], [0.6, 2.0, -3.0, 0.0, 3.0]]),
    # latent engagement: two-state sticky HMM, state-specific bias b_k and cue gain s_k (cf. Ashwood et al. 2022)
    "hmm": (["p11", "p22", "b1", "b2", "s1", "s2", "w"],
            [(0.5, 0.999), (0.5, 0.999), (-14, 14), (-14, 14), (0, 12), (0, 12), (-8, 8)],
            [[0.95, 0.95, -6.0, -2.0, 3.0, 3.0, 0.0], [0.99, 0.9, -8.0, -4.0, 4.0, 2.0, -1.0]]),
    "hmmck": (["p11", "p22", "b1", "b2", "s1", "s2", "w", "rho", "g1"],
              [(0.5, 0.999), (0.5, 0.999), (-14, 14), (-14, 14), (0, 12), (0, 12), (-8, 8), (0.3, 0.999), (-12, 12)],
              [[0.95, 0.95, -6.0, -2.0, 3.0, 3.0, 0.0, 0.85, 1.0], [0.99, 0.9, -8.0, -4.0, 4.0, 2.0, -1.0, 0.6, 2.0]]),
    # value learning PLUS choice retrieval (perseveration): TD or DMA gate on the value term, CK term added to the logit
    "qck": (["alpha", "beta", "rho", "g1", "b", "w", "s"],
            [(0.01, 0.9), (0.2, 15), (0.3, 0.999), (-12, 12), (-14, 14), (-8, 8), (0, 12)],
            [[0.15, 4.0, 0.85, 1.0, -4.0, 0.0, 3.0], [0.4, 8.0, 0.6, 2.0, -7.0, -1.0, 4.0]]),
    "dmack": (["alpha", "beta_base", "lam", "tau_hcn", "rho", "g1", "b", "w", "s"],
              [(0.01, 0.9), (0.2, 15), (0.0, 1.5), (3.0, 300.0), (0.3, 0.999), (-12, 12), (-14, 14), (-8, 8), (0, 12)],
              [[0.15, 5.5, 0.8, 45.0, 0.85, 1.0, -5.0, 0.0, 3.0], [0.3, 8.0, 0.4, 20.0, 0.6, 2.0, -7.0, -1.0, 4.0]]),
    "epi": (["rho", "g1", "g2", "b", "w", "s"], [(0.3, 0.999), (-12, 12), (-12, 12), (-14, 14), (-8, 8), (0, 12)],
            [[0.9, 1.0, -1.0, -4.0, 0.0, 3.0], [0.6, 2.0, 0.0, -3.0, 0.0, 3.0]]),
}


@njit(cache=True)
def _core(mid, P, action, reward, sat, u, rewarded, cue, state, simulate, ll, beta_tr, delta_tr, g_tr):
    n = action.shape[0]
    v0 = 0.33
    V = np.full((2, 2), v0)   # V[state, action]
    g = GMAX
    sm = 0.0
    cum = 0.0
    num_a = np.zeros(2); num_p = np.zeros(2); den = np.zeros(2)   # per-cue episodic traces (models ck, epi)
    for t in range(n):
        s = cum / 100.0 if simulate else sat[t]
        st = state[t]
        V0 = V[st, 0]
        V1 = V[st, 1]
        # ---- policy -------------------------------------------------------------
        if mid == 0:
            beta = 1.0
            z = P[0] + P[1] * s
        elif mid == 1:
            beta = P[1]
            z = beta * (V1 - V0) + P[2] + P[3] * s
        elif mid == 2:
            beta = P[3]
            z = beta * (V1 - V0) + P[4] + P[5] * s
        elif mid == 3:
            beta = P[1] * g / GMAX
            z = beta * (V1 - V0) + P[4] + P[5] * s
        elif mid == 4:
            beta = BETA_C * g / GMAX
            z = beta * (V1 - V0) + P[0] + P[1] * s
        elif mid == 9:
            beta = P[1]
            cc = cue[t]
            ma = num_a[cc] / den[cc] if den[cc] > 0 else 0.0
            z = beta * (V1 - V0) + P[3] * ma + P[4] + P[5] * s
        elif mid == 10:
            beta = P[1] * g / GMAX
            cc = cue[t]
            ma = num_a[cc] / den[cc] if den[cc] > 0 else 0.0
            z = beta * (V1 - V0) + P[5] * ma + P[6] + P[7] * s
        elif mid == 5:
            beta = 1.0
            cc = cue[t]
            ma = num_a[cc] / den[cc] if den[cc] > 0 else 0.0
            z = P[1] * ma + P[2] + P[3] * s
        else:
            beta = 1.0
            cc = cue[t]
            ma = num_a[cc] / den[cc] if den[cc] > 0 else 0.0
            mp = num_p[cc] / den[cc] if den[cc] > 0 else 0.0
            z = P[1] * ma + P[2] * mp + P[3] + P[4] * s
        z += P[P.shape[0] - 1] * cue[t]
        if z > 30.0:
            z = 30.0
        elif z < -30.0:
            z = -30.0
        pl = 1.0 / (1.0 + math.exp(-z))
        if simulate:
            a = 1 if u[t] < pl else 0
            r = 1 if (a == 1 and rewarded[t] == 1) else 0
            action[t] = a
            reward[t] = r
            sat[t] = s
        else:
            a = action[t]
            r = reward[t]
        pa = pl if a == 1 else 1.0 - pl
        if pa < 1e-12:
            pa = 1e-12
        ll[t] = math.log(pa)
        beta_tr[t] = beta
        g_tr[t] = g
        cum += r
        if mid >= 9:
            cc = cue[t]
            rho = P[2] if mid == 9 else P[4]
            num_a[cc] = rho * num_a[cc] + (2.0 * a - 1.0)
            den[cc] = rho * den[cc] + 1.0
        if mid == 0:
            delta_tr[t] = 0.0
            continue
        if mid == 5 or mid == 6:  # store this trial as an engram of its cue and decay the older ones
            cc = cue[t]
            rho = P[0]
            num_a[cc] = rho * num_a[cc] + (2.0 * a - 1.0)
            num_p[cc] = rho * num_p[cc] + (1.0 if (a == 1 and r == 0) else 0.0)
            den[cc] = rho * den[cc] + 1.0
            delta_tr[t] = 0.0
            continue
        # ---- learning -----------------------------------------------------------
        Va = V1 if a == 1 else V0
        delta = r - Va
        delta_tr[t] = delta
        if mid == 1 or mid == 9 or mid == 10:
            a_t = P[0]
        elif mid == 2:
            a_t = P[0] * (1.0 + P[1] * sm)
            if a_t > 0.95:
                a_t = 0.95
            sm = (1.0 - P[2]) * sm + P[2] * abs(delta)
        elif mid == 3:
            a_t = P[0]
        else:
            a_t = ALPHA_C
        Vn = Va + a_t * delta
        if mid == 3 or mid == 4 or mid == 10:  # DMA clips values to [0,1]
            Vn = min(1.0, max(0.0, Vn))
        V[st, a] = Vn
        if mid == 3 or mid == 4 or mid == 10:
            lam = P[2] if (mid == 3 or mid == 10) else LAMBDA_C
            tau = P[3] if (mid == 3 or mid == 10) else TAU_C
            v_ap = VMAX * (1.0 - lam * abs(delta))
            phi = 1.0 / (1.0 + math.exp(-(v_ap - 0.45) / 0.12))
            g = g + (1.0 / tau) * (GMAX * phi - g)
            if g < 0.01:
                g = 0.01
            elif g > GMAX:
                g = GMAX


@njit(cache=True)
def _hmm(P, action, sat, cue, use_ck, ll):
    n = action.shape[0]
    p11 = P[0]; p22 = P[1]
    pi1 = (1.0 - p22) / (2.0 - p11 - p22)
    pr = np.array([pi1, 1.0 - pi1])
    num = np.zeros(2); den = np.zeros(2)
    for t in range(n):
        cc = cue[t]
        ma = 0.0
        if use_ck and den[cc] > 0:
            ma = num[cc] / den[cc]
        pl = 0.0
        pk = np.zeros(2)
        for k in range(2):
            z = P[2 + k] + P[4 + k] * cue[t] + P[6] * sat[t]
            if use_ck:
                z += P[8] * ma
            if z > 30.0: z = 30.0
            elif z < -30.0: z = -30.0
            pk[k] = 1.0 / (1.0 + math.exp(-z))
            pl += pr[k] * pk[k]
        a = action[t]
        pa = pl if a == 1 else 1.0 - pl
        if pa < 1e-12: pa = 1e-12
        ll[t] = math.log(pa)
        po0 = pr[0] * (pk[0] if a == 1 else 1.0 - pk[0])
        po1 = pr[1] * (pk[1] if a == 1 else 1.0 - pk[1])
        sm = po0 + po1
        if sm < 1e-300: sm = 1e-300
        po0 /= sm; po1 /= sm
        pr[0] = po0 * p11 + po1 * (1.0 - p22)
        pr[1] = po0 * (1.0 - p11) + po1 * p22
        if use_ck:
            rho = P[7]
            num[cc] = rho * num[cc] + (2.0 * a - 1.0)
            den[cc] = rho * den[cc] + 1.0


def loglik(model, params, action, reward, sat, cue, state=None, traces=False):
    n = len(action)
    if model in ("hmm", "hmmck"):
        ll = np.empty(n)
        _hmm(np.asarray(params, float), np.array(action, dtype=np.int64), np.asarray(sat, float),
             np.asarray(cue, np.int64), model == "hmmck", ll)
        return ll
    state = np.zeros(n, np.int64) if state is None else np.asarray(state, np.int64)
    ll = np.empty(n); b = np.empty(n); d = np.empty(n); g = np.empty(n)
    _core(MID[model], np.asarray(params, float), np.array(action, dtype=np.int64),
          np.array(reward, dtype=np.int64), np.asarray(sat, float), np.zeros(n),
          np.zeros(n, np.int64), np.asarray(cue, np.int64), state, False, ll, b, d, g)
    return (ll, dict(beta=b, delta=d, g=g)) if traces else ll


def simulate(model, params, rewarded, rng, cue=None, state=None):
    n = len(rewarded)
    cue = rewarded if cue is None else cue
    state = np.zeros(n, np.int64) if state is None else np.asarray(state, np.int64)
    act = np.zeros(n, np.int64); rew = np.zeros(n, np.int64); sat = np.zeros(n)
    ll = np.empty(n); b = np.empty(n); d = np.empty(n); g = np.empty(n)
    _core(MID[model], np.asarray(params, float), act, rew, sat, rng.random(n),
          np.asarray(rewarded, np.int64), np.asarray(cue, np.int64), state, True, ll, b, d, g)
    return act, rew, sat


In [ ]:
%%writefile fit.py
"""Fitting and leave-one-session-out cross-validation (per mouse)."""
from __future__ import annotations

import numpy as np
from scipy.optimize import minimize

import models as M


def nll(params, model, sessions):
    tot = 0.0
    for act, rew, sat, cue, st in sessions:
        tot -= M.loglik(model, params, act, rew, sat, cue, st).sum()
    return tot


def fit(model, sessions, extra_starts=0, rng=None):
    names, bounds, starts = M.SPECS[model]
    starts = [np.array(s, float) for s in starts]
    if extra_starts:
        rng = np.random.default_rng(0) if rng is None else rng
        lo = np.array([b[0] for b in bounds]); hi = np.array([b[1] for b in bounds])
        for _ in range(extra_starts):
            starts.append(lo + rng.random(len(lo)) * (hi - lo))
    best = None
    for s0 in starts:
        r = minimize(nll, s0, args=(model, sessions), method="L-BFGS-B", bounds=bounds)
        if best is None or r.fun < best.fun:
            best = r
    return best.x, best.fun


def loso(mouse_sessions, models_list=None, extra_starts=0):
    """mouse_sessions: list of (act, rew, sat) for ONE mouse. Returns dict model -> array of
    held-out mean log-lik per trial for each session (and the fitted params per fold)."""
    models_list = models_list or M.MODELS
    k = len(mouse_sessions)
    out = {m: np.full(k, np.nan) for m in models_list}
    par = {m: [None] * k for m in models_list}
    if k < 2:
        return out, par
    for i in range(k):
        train = [s for j, s in enumerate(mouse_sessions) if j != i]
        te = mouse_sessions[i]
        for m in models_list:
            p, _ = fit(m, train, extra_starts)
            out[m][i] = M.loglik(m, p, *te).mean()
            par[m][i] = p
    return out, par


def boot_ci(x, n=5000, seed=0):
    x = np.asarray(x, float); x = x[~np.isnan(x)]
    rng = np.random.default_rng(seed)
    bs = rng.choice(x, size=(n, len(x))).mean(1)
    return float(x.mean()), float(np.percentile(bs, 2.5)), float(np.percentile(bs, 97.5))


In [ ]:
%%writefile validate_volatility.py
"""Identifiability of surprise-gated learning as a function of task volatility (SIMULATED mice).

Task "detect": stationary change detection (reward contingency never changes).
Task "switch": two stimuli, only one is rewarded; the rewarded one flips every L trials
               (context switching, as in Allen's Dynamic Routing task).
Truth in {q, dma}. Fitted: q, ph, dma, dma_const (held-out leave-one-session-out likelihood).
Run:  python validate_volatility.py --tasks detect,200,100,50,25 --mice 60 --out vol.json
"""
import argparse, json, sys, time
import numpy as np
import models as M
import fit as F

FIT = ["q", "ph", "dma", "dma_const"]


def draw(model, rng, task):
    w = rng.uniform(-1.0, -.2)
    if task == "detect":
        crit = rng.normal(-1.0, .5); s = rng.uniform(2.5, 5.0)
        b_of = lambda beta: crit - .85 * beta
    else:
        s = 0.0
        b_of = lambda beta: rng.normal(-0.5, .5)
    if model == "q":
        beta = rng.uniform(3, 9)
        return [rng.uniform(.1, .4), beta, b_of(beta), w, s]
    if model == "dma":
        bb = rng.uniform(4, 10)
        return [rng.uniform(.1, .4), bb, rng.uniform(.5, 1.2), rng.uniform(10, 80), b_of(bb), w, s]
    raise ValueError(model)


def make_session(model, params, task, n, rng):
    if task == "detect":
        go = (rng.random(n) > .1).astype(int)
        return M.simulate(model, params, go, rng, cue=go, state=np.zeros(n, int)), go, np.zeros(n, int)
    L = int(task)
    c = rng.integers(0, 2, n)                       # stimulus identity per trial
    blk = (np.arange(n) // L + rng.integers(0, 2)) % 2   # which stimulus is rewarded in this block
    rewarded = (c == blk).astype(int)
    return M.simulate(model, params, rewarded, rng, cue=np.zeros(n, int), state=c), np.zeros(n, int), c


def make_mouse(model, task, rng, n_sessions=4, n_trials=450):
    params = draw(model, rng, task)
    sess = []
    for _ in range(n_sessions):
        (act, rew, sat), cue, st = make_session(model, params, task, n_trials, rng)
        sess.append((act, rew, sat, cue, st))
    return params, sess


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--tasks", default="detect,200,100,50,25")
    ap.add_argument("--mice", type=int, default=60)
    ap.add_argument("--trials", type=int, default=450)
    ap.add_argument("--seed", type=int, default=7)
    ap.add_argument("--out", default="vol.json")
    a = ap.parse_args()
    rng = np.random.default_rng(a.seed)
    res = {}
    t0 = time.time()
    for task in a.tasks.split(","):
        res[task] = {}
        for truth in ["q", "dma"]:
            ll = {m: [] for m in FIT}; wins = {m: 0 for m in FIT}; pr = []
            for k in range(a.mice):
                params, sess = make_mouse(truth, task, rng, n_trials=a.trials)
                out, _ = F.loso(sess, FIT)
                for m in FIT:
                    ll[m].append(float(np.nanmean(out[m])))
                wins[max(FIT, key=lambda m: np.nanmean(out[m]))] += 1
                if truth == "dma":
                    p_hat, _ = F.fit("dma", sess, extra_starts=2, rng=rng)
                    pr.append({"true": list(map(float, params)), "est": list(map(float, p_hat))})
            d = lambda x, y: F.boot_ci(np.array(ll[x]) - np.array(ll[y]))
            res[task][truth] = {"mean_ll": {m: float(np.mean(ll[m])) for m in FIT}, "wins": wins,
                                "dma_minus_q": d("dma", "q"), "ph_minus_q": d("ph", "q"),
                                "dma_minus_ph": d("dma", "ph"), "param_recovery": pr}
            print(task, truth, wins, "dma-q %.4f [%.4f, %.4f]" % res[task][truth]["dma_minus_q"],
                  "%.0fs" % (time.time() - t0), flush=True)
            json.dump(res, open(a.out, "w"))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile prior_sensitivity.py
"""Sensitivity of gate recovery (choices only) to the simulation priors. Truth: DMA. Same fitting as the main analysis.
Variants change the ranges of (alpha, beta_base, lambda, tau).  Usage: python prior_sensitivity.py --mice 40 --out prior_sens.json"""
import argparse, json, numpy as np
import models as M, fit as F, validate_volatility as V

VARIANTS = {  # alpha, beta_base, lam, tau
    "default": ((.1, .4), (4, 10), (.5, 1.2), (10, 80)),
    "strong_fast_gate": ((.1, .4), (6, 12), (1.0, 1.5), (10, 30)),
    "slow_gate": ((.1, .4), (4, 10), (.5, 1.2), (100, 250)),
    "slow_learning": ((.03, .12), (4, 10), (.5, 1.2), (10, 80)),
    "low_beta": ((.1, .4), (1.5, 4), (.5, 1.2), (10, 80)),
}

def main():
    ap = argparse.ArgumentParser(); ap.add_argument("--mice", type=int, default=40); ap.add_argument("--out", default="prior_sens.json")
    ap.add_argument("--tasks", default="detect,50"); a = ap.parse_args(); rng = np.random.default_rng(5); res = {}
    for task in a.tasks.split(","):
        for name, (ra, rb, rl, rt) in VARIANTS.items():
            T, E = [], []
            for _ in range(a.mice):
                crit = rng.normal(-1.0, .5); s = rng.uniform(2.5, 5.0) if task == "detect" else 0.0; w = rng.uniform(-1.0, -.2)
                bb = rng.uniform(*rb); b = crit - .85 * bb if task == "detect" else rng.normal(-0.5, .5)
                p = [rng.uniform(*ra), bb, rng.uniform(*rl), rng.uniform(*rt), b, w, s]
                sess = []
                for _k in range(4):
                    (act, rew, sat), cue, st = V.make_session("dma", p, task, 450, rng); sess.append((act, rew, sat, cue, st))
                ph, _ = F.fit("dma", sess, extra_starts=2, rng=rng); T.append(p); E.append(ph.tolist())
            T = np.array(T); E = np.array(E); r = lambda i: float(np.corrcoef(T[:, i], E[:, i])[0, 1])
            res[f"{task}/{name}"] = {"r_lam": r(2), "r_tau": r(3), "lam_at_bound": float((E[:, 2] < 0.01).mean())}
            print(task, name, res[f"{task}/{name}"], flush=True)
    json.dump(res, open(a.out, "w"), indent=1)
main()


In [ ]:
%%writefile sample_size.py
"""Gate recovery (choices only) as a function of data per animal. Truth: DMA. Usage: python sample_size.py --out sample_size.json"""
import argparse, json, numpy as np
import models as M, fit as F, validate_volatility as V
ap = argparse.ArgumentParser(); ap.add_argument("--mice", type=int, default=40); ap.add_argument("--out", default="sample_size.json")
ap.add_argument("--tasks", default="detect,50"); ap.add_argument("--sizes", default="2x450,4x450,8x450,16x450"); a = ap.parse_args()
rng = np.random.default_rng(31); res = {}
for task in a.tasks.split(","):
    for sz in a.sizes.split(","):
        ns, nt = map(int, sz.split("x")); T, E = [], []
        for _ in range(a.mice):
            p, sess = V.make_mouse("dma", task, rng, n_sessions=ns, n_trials=nt)
            ph, _ = F.fit("dma", sess, extra_starts=2, rng=rng); T.append(p); E.append(ph.tolist())
        T = np.array(T); E = np.array(E); r = lambda i: float(np.corrcoef(T[:, i], E[:, i])[0, 1])
        res[f"{task}/{sz}"] = {"trials_per_animal": ns * nt, "r_lam": r(2), "r_tau": r(3)}
        print(task, sz, res[f"{task}/{sz}"], flush=True)
json.dump(res, open(a.out, "w"), indent=1)


In [ ]:
%%writefile oracle_gap.py
"""Is the DMA's held-out loss to TD (when DMA generates the data) an estimation problem or an information problem?
For each simulated DMA animal: fit TD on 4 training sessions; score 2 NEW sessions from the same animal with
(a) the DMA at its TRUE generating parameters (oracle: no estimation error) and (b) the fitted TD and fitted DMA.
oracle gap = DMA_true - TD_fit on held-out sessions = the most any choice-only DMA analysis could gain here.
Usage: python oracle_gap.py --mice 60 --out oracle_gap.json"""
import argparse, json, numpy as np
import models as M, fit as F, validate_volatility as V

ap = argparse.ArgumentParser(); ap.add_argument("--mice", type=int, default=60); ap.add_argument("--out", default="oracle_gap.json")
ap.add_argument("--tasks", default="detect,200,100,50,25"); a = ap.parse_args(); rng = np.random.default_rng(21); res = {}
for task in a.tasks.split(","):
    orc, fit_d, fit_q = [], [], []
    for _ in range(a.mice):
        params, sess = V.make_mouse("dma", task, rng)
        test = []
        for _k in range(2):
            (act, rew, sat), cue, st = V.make_session("dma", params, task, 450, rng); test.append((act, rew, sat, cue, st))
        pq, _ = F.fit("q", sess, extra_starts=2, rng=rng); pd_, _ = F.fit("dma", sess, extra_starts=2, rng=rng)
        sc = lambda m, p: float(np.mean([M.loglik(m, p, s[0], s[1], s[2], s[3], s[4]).mean() for s in test]))
        q = sc("q", pq); orc.append(sc("dma", np.array(params)) - q); fit_d.append(sc("dma", pd_) - q)
    f = lambda x: [float(np.mean(x) * 1e4)] + [float(v * 1e4) for v in np.percentile(np.array(x), [2.5, 97.5])]
    bs = lambda x: F.boot_ci(np.array(x))
    res[task] = {"oracle_minus_td_e4": [v * 1e4 for v in bs(orc)], "fitted_dma_minus_td_e4": [v * 1e4 for v in bs(fit_d)],
                 "share_oracle_positive": float(np.mean(np.array(orc) > 0))}
    print(task, {k: (np.round(v, 1).tolist() if isinstance(v, list) else round(v, 2)) for k, v in res[task].items()}, flush=True)
json.dump(res, open(a.out, "w"), indent=1)


In [ ]:
%%writefile pupil_power.py
"""Simulation-based power analysis: would a continuous arousal-like observable identify the DMA gate?
ASSUMPTION (not data): the observable is y_t = a*g_t + c*|delta_t| + N(0,1), i.e. it carries the gate trace g_t
with gain a, plus a confounding unsigned-prediction-error term (c = 1) that any joint model must regress out.
Choice-only fit: maximize sum log P(a_t). Joint fit: add the concentrated Gaussian log-likelihood of y
(per animal, regression on [1, g_t, |delta_t|] profiled out). Same simulated animals for both fits.
Usage: python pupil_power.py --task detect --gains 0,1,2,4 --mice 40 --out pupil_detect.json"""
import argparse, json, numpy as np
from scipy.optimize import minimize
import models as M, fit as F, validate_volatility as V

def traces(p, sess):
    out = []
    for act, rew, sat, cue, st in sess:
        ll, tr = M.loglik("dma", p, act, rew, sat, cue, st, traces=True)
        out.append((ll.sum(), tr["g"], np.abs(tr["delta"])))
    return out

def joint_nll(p, sess, ys):
    tr = traces(p, sess)
    nll = -sum(t[0] for t in tr)
    X = np.column_stack([np.ones(sum(len(t[1]) for t in tr)), np.concatenate([t[1] for t in tr]), np.concatenate([t[2] for t in tr])])
    y = np.concatenate(ys); beta = np.linalg.lstsq(X, y, rcond=None)[0]
    rss = float(((y - X @ beta) ** 2).sum()); N = len(y)
    return nll + 0.5 * N * np.log(max(rss / N, 1e-12)), beta

def fit_joint(sess, ys, rng):
    names, bounds, starts = M.SPECS["dma"]
    lo = np.array([b[0] for b in bounds]); hi = np.array([b[1] for b in bounds])
    S = [np.array(s, float) for s in starts] + [lo + rng.random(len(lo)) * (hi - lo) for _ in range(2)]
    best = None
    for s0 in S:
        r = minimize(lambda p: joint_nll(p, sess, ys)[0], s0, method="L-BFGS-B", bounds=bounds)
        if best is None or r.fun < best.fun: best = r
    return best.x

def main():
    ap = argparse.ArgumentParser(); ap.add_argument("--task", default="detect"); ap.add_argument("--gains", default="0,1,2,4")
    ap.add_argument("--mice", type=int, default=40); ap.add_argument("--seed", type=int, default=11); ap.add_argument("--out", default="pupil.json")
    a = ap.parse_args(); rng = np.random.default_rng(a.seed); gains = [float(x) for x in a.gains.split(",")]
    res = {"choice_only": {"true": [], "est": []}}; res.update({str(g): {"true": [], "est": [], "r2g": []} for g in gains})
    for k in range(a.mice):
        params, sess = V.make_mouse("dma", a.task, rng)
        pc, _ = F.fit("dma", sess, extra_starts=2, rng=rng)
        res["choice_only"]["true"].append(list(map(float, params))); res["choice_only"]["est"].append(list(map(float, pc)))
        tr = traces(np.array(params), sess); noise = [rng.standard_normal(len(t[1])) for t in tr]
        for g in gains:
            ys = [g * t[1] + 1.0 * t[2] + e for t, e in zip(tr, noise)]
            var_g = np.var(np.concatenate([g * t[1] for t in tr])); var_y = np.var(np.concatenate(ys))
            pj = fit_joint(sess, ys, rng)
            res[str(g)]["true"].append(list(map(float, params))); res[str(g)]["est"].append(list(map(float, pj))); res[str(g)]["r2g"].append(float(var_g / var_y))
        print(k, flush=True)
    def rec(d, i):
        t = np.array([x[i] for x in d["true"]]); e = np.array([x[i] for x in d["est"]]); return float(np.corrcoef(t, e)[0, 1])
    summ = {k: {"r_lam": rec(v, 2), "r_tau": rec(v, 3), "mean_r2g": float(np.mean(v.get("r2g", [np.nan])))} for k, v in res.items()}
    res["summary"] = summ; json.dump(res, open(a.out, "w")); print(json.dumps(summ, indent=1))
main()


## Ejecución

In [ ]:
!python validate_volatility.py --tasks detect,200,100,50,25 --mice 60 --out vol_a.json

In [ ]:
!python prior_sensitivity.py --mice 40 --out prior_sens.json

In [ ]:
!python sample_size.py --tasks detect --out ss_detect.json
!python sample_size.py --tasks 50 --out ss_50.json

In [ ]:
!python oracle_gap.py --mice 60 --tasks detect,200,100 --out oracle_a.json
!python oracle_gap.py --mice 60 --tasks 50,25 --out oracle_b.json

In [ ]:
!python pupil_power.py --task detect --gains 0,5,10,20 --mice 40 --out pupil_detect.json
!python pupil_power.py --task 50 --gains 0,5,10,20 --mice 40 --out pupil_50.json

In [ ]:
!zip -q sim_results.zip *.json *.csv
from google.colab import files
files.download('sim_results.zip')